In [2]:
# =========================================================
# 베드 grid 폴더 정리/삭제 파이프라인
# 규칙:
#  1차: 엑셀의 bed_keep == 'ㅇ' 인 bed만 보존, 나머지 bed는 폴더 단위 삭제
#  2차: (보존 bed 내) 같은 bed+date에서 time이 가장 이른 폴더만 보존, 나머지 time 폴더 삭제
#  최종: 살아남은 폴더 내부(상추 00~11 png)는 그대로 보존
# ---------------------------------------------------------
# ⚠️ 안전장치:
#  - DRY_RUN=True 기본 (실제 삭제 안 함)
#  - 실제 삭제는 DRY_RUN=False로 바꾸고 실행
#  - 기본은 "삭제"가 아니라 "휴지통 폴더로 이동"(MOVE_TO_TRASH=True)
# =========================================================
# =========================================================
# 1차 삭제: bed_keep 기준으로 bed 단위 폴더 정리
# ---------------------------------------------------------
# 규칙:
#  - 엑셀의 bed_keep == 'ㅇ' 인 bed만 보존
#  - bed_keep != 'ㅇ' 인 bed는 "폴더 단위"로 이동/삭제
# ---------------------------------------------------------
# ⚠️ 안전장치:
#  - DRY_RUN=True 기본 (실제 이동/삭제 안 함)
#  - MOVE_TO_TRASH=True 기본 (삭제 대신 _TRASH 로 이동)
#  - 실행 전 반드시 _delete_plan_stage1.csv 확인
# =========================================================

import os
import re
import shutil
from pathlib import Path
import pandas as pd

# =============================
# 사용자 설정
# =============================
BASE_DIR = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/3. ROI_LETTUCE/bed_grid_split/251207-251212")
EXCEL_PATH = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/4. 결과 출력 시각화/1작기(251128-251226)/1파트(251128-251213)/251212_lettuce_segmentation_area(2)_index.xlsx")
SHEET_NAME = "Sheet1"

BED_KEEP_COL = "bed_keep"   # bed_keep 열 이름
KEEP_MARK = "ㅇ"            # 살림 표시

# 날짜 범위 폴더(예: 251129-251206)를 지정하면 그 하위만 처리. 빈 문자열이면 BASE_DIR 전체.
DATE_RANGE_FOLDER = ""   # "" 로 두면 BASE_DIR 전체 탐색

# 실행 옵션
DRY_RUN = False             # True: 실제 이동/삭제 안 함
MOVE_TO_TRASH = False        # True: 삭제 대신 TRASH_DIR로 이동
TRASH_DIR = BASE_DIR / "_TRASH_STAGE1"

# 로그/리포트 저장
REPORT_CSV = BASE_DIR / "_delete_plan_stage1.csv"

# =============================
# 유틸: 폴더명 파싱
# =============================
# 기대 폴더명: bed00_20251204_102819_cam0
FOLDER_RE = re.compile(r"^(?P<bed>bed\d{2})_(?P<date>\d{8})_(?P<time>\d{6})_(?P<cam>cam\d+)$")

def parse_folder_name(name: str):
    m = FOLDER_RE.match(name)
    if not m:
        return None
    d = m.groupdict()
    return d["bed"], d["date"], d["time"], d["cam"]


def ensure_parent_name_col(df: pd.DataFrame) -> pd.DataFrame:
    """엑셀에 parent_name(=최종 하위폴더명) 컬럼이 없으면 만들어줌.
    우선순위: parent_name 있으면 사용, 없으면 image_path에서 폴더명 추출, 없으면 filename에서 parent_name 생성.
    """
    if "parent_name" in df.columns:
        return df

    if "image_path" in df.columns:
        df["parent_name"] = df["image_path"].astype(str).apply(lambda x: Path(x).parent.name)
        return df

    if "filename" in df.columns:
        # filename: bed00_20251205_094145_cam0_06.png
        # parent_name: bed00_20251205_094145_cam0
        df["parent_name"] = df["filename"].astype(str).apply(lambda x: "_".join(Path(x).stem.split("_")[:4]))
        return df

    raise ValueError("엑셀에서 parent_name/image_path/filename 중 하나가 필요합니다.")


def load_excel() -> pd.DataFrame:
    df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)
    if BED_KEEP_COL not in df.columns:
        raise ValueError(f"엑셀에 '{BED_KEEP_COL}' 열이 없습니다. (현재 열 일부: {list(df.columns)[:30]})")

    df = ensure_parent_name_col(df)

    parsed = df["parent_name"].astype(str).apply(parse_folder_name)
    ok = parsed.notna()
    df = df.loc[ok].copy()
    df[["bed", "date", "time", "cam"]] = pd.DataFrame(parsed[ok].tolist(), index=df.index)
    return df


def collect_target_folders(base_dir: Path):
    search_root = base_dir
    if DATE_RANGE_FOLDER:
        search_root = base_dir / DATE_RANGE_FOLDER

    if not search_root.exists():
        print(f"[WARN] 검색 루트 폴더 없음: {search_root}")
        return []

    folders = []
    # Use rglob to perform a recursive search for all directories under search_root.
    # Then filter these by the FOLDER_RE pattern.
    for p in search_root.rglob('*'):
        if p.is_dir() and FOLDER_RE.match(p.name):
            folders.append(p)
    return sorted(folders)


def plan_stage1(df: pd.DataFrame, folders_on_disk: list[Path]) -> pd.DataFrame:
    # bed_keep 맵: bed 내에서 KEEP_MARK가 하나라도 있으면 살림
    bed_keep_map = (
        df.groupby("bed")[BED_KEEP_COL]
          .apply(lambda s: KEEP_MARK in set(map(str, s)))
          .to_dict()
    )
    keep_beds = {b for b, keep in bed_keep_map.items() if keep}

    # 디스크 폴더 메타
    disk_meta = []
    for p in folders_on_disk:
        parts = parse_folder_name(p.name)
        if parts is None:
            continue
        bed, date, time, cam = parts
        disk_meta.append({"folder": str(p), "bed": bed, "date": date, "time": time, "cam": cam})

    disk_df = pd.DataFrame(disk_meta)
    if disk_df.empty:
        return disk_df

    disk_df["keep_bed"] = disk_df["bed"].isin(keep_beds)
    disk_df["keep_final"] = disk_df["keep_bed"]
    disk_df["reason"] = disk_df.apply(lambda r: "KEEP" if r["keep_final"] else "bed_keep=X (bed 단위 제외)", axis=1)

    return disk_df


def execute_stage1(plan_df: pd.DataFrame):
    # Add this line to display the head of the DataFrame before saving
    print("[INFO] plan_df 내용 (상위 5행):")
    display(plan_df.head())

    plan_df.to_csv(REPORT_CSV, index=False, encoding="utf-8-sig")
    print(f"[INFO] 1차 삭제 계획 저장: {REPORT_CSV}")

    if plan_df.empty:
        print("[INFO] 삭제/이동할 폴더가 없습니다.")
        return

    to_remove = plan_df[plan_df["keep_final"] == False].copy()
    folders = [Path(p) for p in to_remove["folder"].tolist()]

    print(f"[INFO] 1차 이동/삭제 대상 폴더 수: {len(folders)}")

    if DRY_RUN:
        print("[DRY_RUN] 실제 이동/삭제는 수행하지 않습니다. 상위 20개만 출력:")
        for p in folders[:20]:
            print(" -", p)
        return

    if MOVE_TO_TRASH:
        TRASH_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[INFO] TRASH_DIR: {TRASH_DIR}")

    for p in folders:
        try:
            if MOVE_TO_TRASH:
                dest = TRASH_DIR / p.name
                if dest.exists():
                    dest = TRASH_DIR / f"{p.name}__dup"
                shutil.move(str(p), str(dest))
            else:
                shutil.rmtree(p)
        except Exception as e:
            print(f"[ERROR] 실패: {p} -> {e}")


def main():
    df = load_excel()
    folders_on_disk = collect_target_folders(BASE_DIR)

    print(f"[INFO] 엑셀 rows: {len(df)}")
    print(f"[INFO] 디스크 최종 하위폴더 수: {len(folders_on_disk)}")

    plan_df = plan_stage1(df, folders_on_disk)
    execute_stage1(plan_df)


if __name__ == "__main__":
    main()


[INFO] 엑셀 rows: 39546
[INFO] 디스크 최종 하위폴더 수: 1878
[INFO] plan_df 내용 (상위 5행):


,folder,bed,date,time,cam,keep_bed,keep_final,reason
0,/content/drive/Othercomputers/내 컴퓨터/새 포...,bed00,20251207,072520,cam0,False,False,bed_keep=X (bed 단위 제외)
1,/content/drive/Othercomputers/내 컴퓨터/새 포...,bed00,20251207,072520,cam1,False,False,bed_keep=X (bed 단위 제외)
2,/content/drive/Othercomputers/내 컴퓨터/새 포...,bed00,20251207,112137,cam1,False,False,bed_keep=X (bed 단위 제외)
3,/content/drive/Othercomputers/내 컴퓨터/새 포...,bed00,20251207,202534,cam0,False,False,bed_keep=X (bed 단위 제외)
4,/content/drive/Othercomputers/내 컴퓨터/새 포...,bed00,20251207,212728,cam0,False,False,bed_keep=X (bed 단위 제외)


[INFO] 1차 삭제 계획 저장: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/3. ROI_LETTUCE/bed_grid_split/251207-251212/_delete_plan_stage1.csv
[INFO] 1차 이동/삭제 대상 폴더 수: 1694


##2차 삭제

In [4]:
# =========================================================
# 2차 삭제: 같은 bed+date 내에서 "최초 time" 폴더만 보존
# ---------------------------------------------------------
# 전제:
#  - 1차 삭제(=bed_keep 기준)가 끝난 상태에서 실행 권장
#  - 즉, BASE_DIR 하위에는 이미 keep bed 위주로 남아있을 것
# ---------------------------------------------------------
# 규칙:
#  - 같은 bed+date 그룹에서 time이 가장 작은 폴더만 keep
#  - 나머지 time 폴더는 "폴더 단위"로 이동/삭제
# ---------------------------------------------------------
# ⚠️ 안전장치:
#  - DRY_RUN=True 기본 (실제 이동/삭제 안 함)
#  - MOVE_TO_TRASH=True 기본 (삭제 대신 _TRASH 로 이동)
#  - 실행 전 반드시 _delete_plan_stage2.csv 확인
# =========================================================

import re
import shutil
from pathlib import Path
import pandas as pd

# =============================
# 사용자 설정
# =============================
BASE_DIR = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/3. ROI_LETTUCE/bed_grid_split/251207-251212")

# 날짜 범위 폴더(예: 251129-251206)를 지정하면 그 하위만 처리. 빈 문자열이면 BASE_DIR 전체.
DATE_RANGE_FOLDER = ""   # "" 로 두면 BASE_DIR 전체 탐색

# 실행 옵션
DRY_RUN = False             # True: 실제 이동/삭제 안 함
MOVE_TO_TRASH = False       # True: 삭제 대신 TRASH_DIR로 이동
TRASH_DIR = BASE_DIR / "_TRASH_STAGE2"

# 로그/리포트 저장
REPORT_CSV = BASE_DIR / "_delete_plan_stage2.csv"

# =============================
# 유틸: 폴더명 파싱
# =============================
# 기대 폴더명: bed00_20251204_102819_cam0
FOLDER_RE = re.compile(r"^(?P<bed>bed\d{2})_(?P<date>\d{8})_(?P<time>\d{6})_(?P<cam>cam\d+)$")

def parse_folder_name(name: str):
    m = FOLDER_RE.match(name)
    if not m:
        return None
    d = m.groupdict()
    return d["bed"], d["date"], d["time"], d["cam"]


def collect_target_folders(base_dir: Path):
    roots = [base_dir]
    if DATE_RANGE_FOLDER:
        roots = [base_dir / DATE_RANGE_FOLDER]

    folders = []
    for r in roots:
        if not r.exists():
            print(f"[WARN] 폴더 없음: {r}")
            continue
        for p in r.iterdir():
            if p.is_dir() and FOLDER_RE.match(p.name):
                folders.append(p)
    return sorted(folders)


def plan_stage2(folders_on_disk: list[Path]) -> pd.DataFrame:
    disk_meta = []
    for p in folders_on_disk:
        parts = parse_folder_name(p.name)
        if parts is None:
            continue
        bed, date, time, cam = parts
        disk_meta.append({"folder": str(p), "bed": bed, "date": date, "time": time, "cam": cam})

    disk_df = pd.DataFrame(disk_meta)
    if disk_df.empty:
        return disk_df

    # 같은 bed+date에서 최소 time만 keep
    disk_df["min_time"] = disk_df.groupby(["bed", "date"])['time'].transform('min')
    disk_df["keep_final"] = disk_df['time'].eq(disk_df['min_time'])

    disk_df["reason"] = disk_df.apply(
        lambda r: "KEEP" if r["keep_final"] else "동일 bed+date 내 최초시간 아님",
        axis=1
    )

    return disk_df


def execute_stage2(plan_df: pd.DataFrame):
    plan_df.to_csv(REPORT_CSV, index=False, encoding="utf-8-sig")
    print(f"[INFO] 2차 삭제 계획 저장: {REPORT_CSV}")

    to_remove = plan_df[plan_df["keep_final"] == False].copy()
    folders = [Path(p) for p in to_remove["folder"].tolist()]

    print(f"[INFO] 2차 이동/삭제 대상 폴더 수: {len(folders)}")

    if DRY_RUN:
        print("[DRY_RUN] 실제 이동/삭제는 수행하지 않습니다. 상위 20개만 출력:")
        for p in folders[:20]:
            print(" -", p)
        return

    if MOVE_TO_TRASH:
        TRASH_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[INFO] TRASH_DIR: {TRASH_DIR}")

    for p in folders:
        try:
            if MOVE_TO_TRASH:
                dest = TRASH_DIR / p.name
                if dest.exists():
                    dest = TRASH_DIR / f"{p.name}__dup"
                shutil.move(str(p), str(dest))
            else:
                shutil.rmtree(p)
        except Exception as e:
            print(f"[ERROR] 실패: {p} -> {e}")


def main():
    folders_on_disk = collect_target_folders(BASE_DIR)
    print(f"[INFO] 디스크 최종 하위폴더 수: {len(folders_on_disk)}")

    plan_df = plan_stage2(folders_on_disk)
    execute_stage2(plan_df)


if __name__ == "__main__":
    main()


[INFO] 디스크 최종 하위폴더 수: 184
[INFO] 2차 삭제 계획 저장: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/3. ROI_LETTUCE/bed_grid_split/251207-251212/_delete_plan_stage2.csv
[INFO] 2차 이동/삭제 대상 폴더 수: 130


###원본데이터에서도 삭제하기: 1차

In [7]:
# =========================================================
# [원본 데이터용] 1차 삭제: bed_keep 기준으로 '이미지 파일' 정리
# ---------------------------------------------------------
# ✅ 원본 폴더 구조(예)
#   RAW_ROOT/
#     251128/
#       bed01_20251128_183832_cam0.jpg
#       bed01_20251128_183832_cam1.jpg
#     251129/
#       ...
#
# ✅ 엑셀 구조(예)
#   - filename: bed00_20251204_102819_cam0_06.png   (상추번호/확장자 포함)
#   - bed_keep: 'ㅇ' / 'X'  (이미 존재)
#
# ✅ 매칭 규칙(핵심)
#   - 엑셀 filename에서 '베드단위 키'만 추출:
#       bed00_20251204_102819_cam0   (앞 4토큰: bedXX + 날짜8 + 시간6 + camN)
#   - 원본 이미지 파일명(확장자 제거):
#       bed01_20251128_183832_cam0
#   - 둘을 대조해서 KEEP/DELETE 결정
#
# ✅ 동작
#   - 엑셀에서 bed_keep == 'ㅇ' 인 키만 KEEP
#   - 원본 폴더를 재귀 탐색하면서
#       · 키가 KEEP에 있으면 보존
#       · 없으면 이동/삭제
#
# ⚠️ 안전장치
#   - DRY_RUN=True 기본 (실제 이동/삭제 안 함)
#   - MOVE_TO_TRASH=True 기본 (삭제 대신 _TRASH 로 이동)
#   - 실행 전 _delete_plan_stage1_raw.csv 확인
# =========================================================

import os
import shutil
from pathlib import Path
import pandas as pd

# =============================
# 사용자 설정
# =============================
RAW_ROOT = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251207-251212")  # ✅ 원본 데이터 최상위 폴더
EXCEL_PATH = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/4. 결과 출력 시각화/1작기(251128-251226)/1파트(251128-251213)/251212_lettuce_segmentation_area(2)_index.xlsx")
SHEET_NAME = "Sheet1"

FILENAME_COL = "filename"   # 엑셀 파일명 컬럼
BED_KEEP_COL = "bed_keep"   # 엑셀 bed_keep 컬럼(이미 존재)
KEEP_MARK = "ㅇ"

# 실행 옵션
DRY_RUN = False
MOVE_TO_TRASH = False
TRASH_DIR = RAW_ROOT / "_TRASH_STAGE1_RAW"  # 원본 루트 하단에 휴지통 역할 폴더

# 로그/리포트 저장
REPORT_CSV = RAW_ROOT / "_delete_plan_stage1_raw.csv"

# 처리할 확장자(원본에 jpg/png 섞이면 여기 추가)
ALLOW_EXT = {".jpg", ".jpeg", ".png"}


# =============================
# 유틸: 엑셀 filename -> 베드단위 키 추출
# =============================
# 예) bed00_20251204_102819_cam0_06.png -> bed00_20251204_102819_cam0
# 예) bed00_20251204_102819_cam0.png    -> bed00_20251204_102819_cam0

def excel_to_bed_key(fn: str) -> str:
    stem = Path(str(fn)).stem  # 확장자 제거
    parts = stem.split("_")
    # 최소 bedXX_YYYYMMDD_HHMMSS_camN 까지는 있어야 함
    if len(parts) < 4:
        return ""
    return "_".join(parts[:4])


def load_keep_keys() -> set:
    df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)
    if FILENAME_COL not in df.columns:
        raise ValueError(f"엑셀에 '{FILENAME_COL}' 컬럼이 없습니다. 현재 컬럼 일부: {list(df.columns)[:30]}")
    if BED_KEEP_COL not in df.columns:
        raise ValueError(f"엑셀에 '{BED_KEEP_COL}' 컬럼이 없습니다. 현재 컬럼 일부: {list(df.columns)[:30]}")

    df_keep = df[df[BED_KEEP_COL].astype(str) == KEEP_MARK].copy()
    keys = df_keep[FILENAME_COL].astype(str).apply(excel_to_bed_key)
    keys = keys[keys != ""]

    keep_keys = set(keys.tolist())
    print(f"[INFO] 엑셀 KEEP 키 개수(bed_keep=='ㅇ'): {len(keep_keys)}")
    return keep_keys


def iter_raw_images(root: Path):
    # RAW_ROOT 아래 날짜폴더들이 있지만, 그냥 재귀 탐색으로 다 긁음
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in ALLOW_EXT:
            yield p


def make_plan(keep_keys: set) -> pd.DataFrame:
    rows = []
    for img_path in iter_raw_images(RAW_ROOT):
        key = img_path.stem  # bed01_20251128_183832_cam0
        keep = key in keep_keys
        rows.append({
            "file": str(img_path),
            "key": key,
            "keep_final": bool(keep),
            "reason": "KEEP" if keep else "bed_keep!=ㅇ (원본 이미지 제거 대상)",
        })

    plan_df = pd.DataFrame(rows)
    print(f"[INFO] 원본 이미지 파일 수: {len(plan_df)}")
    if len(plan_df) > 0:
        print(f"[INFO] KEEP 수: {(plan_df['keep_final'] == True).sum()} / REMOVE 수: {(plan_df['keep_final'] == False).sum()}")
    return plan_df


def execute(plan_df: pd.DataFrame):
    plan_df.to_csv(REPORT_CSV, index=False, encoding="utf-8-sig")
    print(f"[INFO] 삭제 계획 저장: {REPORT_CSV}")

    to_remove = plan_df[plan_df["keep_final"] == False].copy()
    files = [Path(p) for p in to_remove["file"].tolist()]

    print(f"[INFO] 이동/삭제 대상 파일 수: {len(files)}")

    if DRY_RUN:
        print("[DRY_RUN] 실제 이동/삭제는 수행하지 않습니다. 상위 30개만 출력:")
        for p in files[:30]:
            print(" -", p)
        return

    if MOVE_TO_TRASH:
        TRASH_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[INFO] TRASH_DIR: {TRASH_DIR}")

    for p in files:
        try:
            if not p.exists():
                continue

            if MOVE_TO_TRASH:
                # 날짜 폴더 구조를 유지해서 이동 (복구 편하게)
                rel = p.relative_to(RAW_ROOT)
                dest = TRASH_DIR / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(p), str(dest))
            else:
                p.unlink()  # ✅ 휴지통 없이 즉시 삭제
        except Exception as e:
            print(f"[ERROR] 실패: {p} -> {e}")


def main():
    keep_keys = load_keep_keys()

    # 안전: keep_keys가 너무 적거나 0이면 바로 멈추게(실수 방지)
    if len(keep_keys) == 0:
        raise RuntimeError("KEEP 키가 0개입니다. bed_keep 표기/엑셀 경로/시트 확인하세요.")

    plan_df = make_plan(keep_keys)
    execute(plan_df)


if __name__ == "__main__":
    main()


[INFO] 엑셀 KEEP 키 개수(bed_keep=='ㅇ'): 184
[INFO] 원본 이미지 파일 수: 7298
[INFO] KEEP 수: 184 / REMOVE 수: 7114
[INFO] 삭제 계획 저장: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251207-251212/_delete_plan_stage1_raw.csv
[INFO] 이동/삭제 대상 파일 수: 7114


##2차 삭제

In [10]:
# =========================================================
# [원본 데이터용] 2차 삭제: 같은 bed+date 내에서 "최초 time" 이미지(파일)만 보존
# ---------------------------------------------------------
# ✅ 전제
#   - 1차 삭제(bed_keep 기준)가 끝난 상태에서 실행 권장
#   - 원본 폴더 구조(예)
#       RAW_ROOT/
#         251128/
#           bed01_20251128_183832_cam0.jpg
#           bed01_20251128_214427_cam0.jpg
#         251129/
#           ...
#
# ✅ 규칙
#   - key = bedXX_YYYYMMDD_HHMMSS_camN  (확장자 제거)
#   - 그룹 = (bed, date, cam)
#   - 각 그룹에서 time이 가장 작은(최초) 파일만 KEEP
#   - 나머지는 이동/삭제
#
# ⚠️ 안전장치
#   - DRY_RUN=True 기본 (실제 이동/삭제 안 함)
#   - MOVE_TO_TRASH=True 기본 (삭제 대신 _TRASH 로 이동)
#   - 실행 전 _delete_plan_stage2_raw.csv 확인
# =========================================================

import re
import shutil
from pathlib import Path
import pandas as pd

# =============================
# 사용자 설정
# =============================
RAW_ROOT = Path(r"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251207-251212")

# 실행 옵션
DRY_RUN = False              # True: 실제 이동/삭제 안 함
MOVE_TO_TRASH = False        # True: 삭제 대신 TRASH_DIR로 이동
TRASH_DIR = RAW_ROOT / "_TRASH_STAGE2_RAW"

# 로그/리포트 저장
REPORT_CSV = RAW_ROOT / "_delete_plan_stage2_raw.csv"

# 처리할 확장자
ALLOW_EXT = {".jpg", ".jpeg", ".png"}

# =============================
# 유틸: 파일명 파싱
# =============================
# 기대 파일명(stem): bed01_20251128_183832_cam0
# 그룹 기준: (bed, date, cam)  => 같은 날/같은 cam에서 최초 time만 유지
FILE_RE = re.compile(r"^(?P<bed>bed\d{2})_(?P<date>\d{8})_(?P<time>\d{6})_(?P<cam>cam\d+)$")

def parse_stem(stem: str):
    m = FILE_RE.match(stem)
    if not m:
        return None
    d = m.groupdict()
    return d["bed"], d["date"], d["time"], d["cam"]


def iter_raw_images(root: Path):
    # RAW_ROOT 아래 날짜폴더들이 있지만, 재귀 탐색으로 다 긁음
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in ALLOW_EXT:
            yield p


def build_plan() -> pd.DataFrame:
    rows = []
    for p in iter_raw_images(RAW_ROOT):
        meta = parse_stem(p.stem)
        if meta is None:
            # 파싱 실패 파일은 건드리지 않음(안전)
            rows.append({
                "file": str(p),
                "bed": None,
                "date": None,
                "time": None,
                "cam": None,
                "keep_final": True,
                "reason": "KEEP (패턴 불일치: 안전상 제외)",
            })
            continue

        bed, date, time, cam = meta
        rows.append({
            "file": str(p),
            "bed": bed,
            "date": date,
            "time": time,
            "cam": cam,
            "keep_final": None,
            "reason": "",
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # 패턴에 맞는 파일들만 대상으로 최초 time 선택
    mask = df["bed"].notna()
    df_valid = df[mask].copy()

    # 그룹: bed+date+cam
    df_valid["min_time"] = df_valid.groupby(["bed", "date", "cam"])["time"].transform("min")
    df_valid["keep_final"] = df_valid["time"].eq(df_valid["min_time"])
    df_valid["reason"] = df_valid.apply(
        lambda r: "KEEP" if r["keep_final"] else "동일 bed+date+cam 내 최초시간 아님",
        axis=1,
    )

    # 원본 df에 반영
    df.loc[mask, "keep_final"] = df_valid["keep_final"].values
    df.loc[mask, "reason"] = df_valid["reason"].values

    # min_time 컬럼은 리포트에 남겨도 좋음
    df["min_time"] = None
    df.loc[mask, "min_time"] = df_valid["min_time"].values

    return df


def execute(plan_df: pd.DataFrame):
    plan_df.to_csv(REPORT_CSV, index=False, encoding="utf-8-sig")
    print(f"[INFO] 2차 삭제 계획 저장: {REPORT_CSV}")

    # 제거 대상(keep_final == False)
    to_remove = plan_df[plan_df["keep_final"] == False].copy()
    files = [Path(p) for p in to_remove["file"].tolist()]

    print(f"[INFO] 원본 이미지 파일 수: {len(plan_df)}")
    print(
        f"[INFO] KEEP 수: {(plan_df['keep_final'] == True).sum()} / REMOVE 수: {(plan_df['keep_final'] == False).sum()}"
    )
    print(f"[INFO] 2차 이동/삭제 대상 파일 수: {len(files)}")

    if DRY_RUN:
        print("[DRY_RUN] 실제 이동/삭제는 수행하지 않습니다. 상위 30개만 출력:")
        for p in files[:30]:
            print(" -", p)
        return

    if MOVE_TO_TRASH:
        TRASH_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[INFO] TRASH_DIR: {TRASH_DIR}")

    for p in files:
        try:
            if not p.exists():
                continue

            if MOVE_TO_TRASH:
                # 날짜 폴더 구조 유지
                rel = p.relative_to(RAW_ROOT)
                dest = TRASH_DIR / rel
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.move(str(p), str(dest))
            else:
                p.unlink()  # 휴지통 없이 즉시 삭제
        except Exception as e:
            print(f"[ERROR] 실패: {p} -> {e}")


def main():
    plan_df = build_plan()
    if plan_df.empty:
        print("[WARN] 처리할 파일이 없습니다.")
        return

    execute(plan_df)


if __name__ == "__main__":
    main()


[INFO] 2차 삭제 계획 저장: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/0. 원본/251207-251212/_delete_plan_stage2_raw.csv
[INFO] 원본 이미지 파일 수: 72
[INFO] KEEP 수: 72 / REMOVE 수: 0
[INFO] 2차 이동/삭제 대상 파일 수: 0
